In [5]:
import requests
import pandas as pd
import datetime


In [6]:
import requests
import pandas as pd
import time

API_KEY = "aa6877ac64bbbc776b89c98b61b11b54"
lat = 24.8607
lon = 67.0011

# Define start and end dates
end_date = pd.to_datetime("today").normalize()
start_date = end_date - pd.DateOffset(years=1)

# Convert to timestamps
start_ts = int(start_date.timestamp())
end_ts = int(end_date.timestamp())

# OpenWeatherMap allows limited range per request, so chunk by 30 days
chunk_days = 30
chunk_seconds = chunk_days * 24 * 60 * 60

all_data = []

current_start = start_ts
while current_start < end_ts:
    current_end = min(current_start + chunk_seconds, end_ts)
    
    params = {
        'lat': lat,
        'lon': lon,
        'start': current_start,
        'end': current_end,
        'appid': API_KEY
    }
    
    response = requests.get("http://api.openweathermap.org/data/2.5/air_pollution/history", params=params)
    response.raise_for_status()
    data = response.json()
    
    if 'list' in data:
        all_data.extend(data['list'])
    
    print(f"Fetched data from {current_start} to {current_end}, total records: {len(data.get('list', []))}")
    current_start = current_end + 1
    time.sleep(1)  # avoid hitting rate limits

# Convert to DataFrame



Fetched data from 1738195200 to 1740787200, total records: 721
Fetched data from 1740787201 to 1743379201, total records: 624
Fetched data from 1743379202 to 1745971202, total records: 576
Fetched data from 1745971203 to 1748563203, total records: 720
Fetched data from 1748563204 to 1751155204, total records: 720
Fetched data from 1751155205 to 1753747205, total records: 720
Fetched data from 1753747206 to 1756339206, total records: 720
Fetched data from 1756339207 to 1758931207, total records: 720
Fetched data from 1758931208 to 1761523208, total records: 720
Fetched data from 1761523209 to 1764115209, total records: 720
Fetched data from 1764115210 to 1766707210, total records: 720
Fetched data from 1766707211 to 1769299211, total records: 696
Fetched data from 1769299212 to 1769731200, total records: 120


In [7]:
raw_df = pd.DataFrame(all_data)
raw_df['datetime'] = pd.to_datetime(raw_df['dt'], unit='s')
raw_df.head()

,main,components,dt,datetime
0,{'aqi': 4},"{'co': 801.09, 'no': 0, 'no2': 19.19, 'o3': 59...",1738195200,2025-01-30 00:00:00
1,{'aqi': 5},"{'co': 988.01, 'no': 0, 'no2': 26.05, 'o3': 46...",1738198800,2025-01-30 01:00:00
2,{'aqi': 5},"{'co': 1548.77, 'no': 0.11, 'no2': 46.61, 'o3'...",1738202400,2025-01-30 02:00:00
3,{'aqi': 5},"{'co': 3097.53, 'no': 19.22, 'no2': 80.2, 'o3'...",1738206000,2025-01-30 03:00:00
4,{'aqi': 5},"{'co': 5447.39, 'no': 79.57, 'no2': 97.33, 'o3...",1738209600,2025-01-30 04:00:00


In [8]:
df_main = raw_df['main'].apply(pd.Series)
df_components = raw_df['components'].apply(pd.Series)
df_components.head()

,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,801.09,0.00,19.19,59.37,10.01,69.45,150.00,6.90
1,988.01,0.00,26.05,46.49,12.40,76.61,164.36,8.36
2,1548.77,0.11,46.61,25.75,17.17,100.83,192.46,14.19
3,3097.53,19.22,80.20,1.61,27.66,174.18,272.99,31.92
4,5447.39,79.57,97.33,4.56,41.96,278.51,386.64,58.77


In [9]:
df = pd.concat([raw_df.drop(['main', 'components'], axis=1), df_main, df_components], axis=1)
df.head()

,dt,datetime,aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,1738195200,2025-01-30 00:00:00,4.0,801.09,0.00,19.19,59.37,10.01,69.45,150.00,6.90
1,1738198800,2025-01-30 01:00:00,5.0,988.01,0.00,26.05,46.49,12.40,76.61,164.36,8.36
2,1738202400,2025-01-30 02:00:00,5.0,1548.77,0.11,46.61,25.75,17.17,100.83,192.46,14.19
3,1738206000,2025-01-30 03:00:00,5.0,3097.53,19.22,80.20,1.61,27.66,174.18,272.99,31.92
4,1738209600,2025-01-30 04:00:00,5.0,5447.39,79.57,97.33,4.56,41.96,278.51,386.64,58.77


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8497 entries, 0 to 8496
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   dt        8497 non-null   int64         
 1   datetime  8497 non-null   datetime64[ns]
 2   aqi       8497 non-null   float64       
 3   co        8497 non-null   float64       
 4   no        8497 non-null   float64       
 5   no2       8497 non-null   float64       
 6   o3        8497 non-null   float64       
 7   so2       8497 non-null   float64       
 8   pm2_5     8497 non-null   float64       
 9   pm10      8497 non-null   float64       
 10  nh3       8497 non-null   float64       
dtypes: datetime64[ns](1), float64(9), int64(1)
memory usage: 730.3 KB


In [11]:
df.sort_values('datetime', inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

,dt,datetime,aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,1738195200,2025-01-30 00:00:00,4.0,801.09,0.00,19.19,59.37,10.01,69.45,150.00,6.90
1,1738198800,2025-01-30 01:00:00,5.0,988.01,0.00,26.05,46.49,12.40,76.61,164.36,8.36
2,1738202400,2025-01-30 02:00:00,5.0,1548.77,0.11,46.61,25.75,17.17,100.83,192.46,14.19
3,1738206000,2025-01-30 03:00:00,5.0,3097.53,19.22,80.20,1.61,27.66,174.18,272.99,31.92
4,1738209600,2025-01-30 04:00:00,5.0,5447.39,79.57,97.33,4.56,41.96,278.51,386.64,58.77


In [12]:
df.to_csv("../Dataset/aqi_data.csv", index=False)

In [13]:
df['aqi'].value_counts()


aqi
3.000000    3485
4.000000    2203
2.000000    1217
5.000000    1124
1.000000     466
3.333332       1
5.555580       1
Name: count, dtype: int64